# Data Science 실전 문제풀이 Set 01~06

이 노트북은 업로드된 `problem.md`와 6개 CSV 데이터셋을 기준으로 **문제를 단계별로 풀고**, 처음 보는 사람도 따라올 수 있도록 설명을 붙인 풀이 노트북입니다.

## 사용 방법

1. 이 노트북 파일과 CSV 파일들을 같은 폴더에 두고 실행하면 됩니다.
2. CSV 파일이 `dataset` 폴더 안에 있다면, 노트북이 자동으로 `dataset` 폴더도 찾습니다.
3. 그래도 파일을 못 찾으면 아래 `DATA_DIR` 후보 경로 부분에서 직접 경로를 수정하세요.

## 최종 답안 요약

| 세트 | Q01 | Q02 | Q03 |
|---|---:|---:|---|
| Set 01 | 510 | 0.38 | 할인율 |
| Set 02 | 3.4 | 0.999 | 0.55 |
| Set 03 | 11.01 | 0.95 | 3 |
| Set 04 | 4 | 0.13 | 0.18 |
| Set 05 | 0.95 | 3946.19 | 1039.19 / 정답표 표기 1039.2 |
| Set 06 | 1.77 | 1.67 | 0.71 |

# 0. 공통 준비

먼저 필요한 라이브러리를 불러옵니다.

- `pandas`, `numpy`: 데이터 처리
- `re`: 문자열에서 `60 Hz`, `4K` 같은 패턴 찾기
- `sklearn`: 머신러닝 모델, 정규화, 평가 지표

또한 CSV 파일 위치를 자동으로 찾는 코드를 넣었습니다.

In [1]:
from pathlib import Path
import re
import math
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, mean_squared_error, silhouette_score, accuracy_score
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeRegressor

pd.set_option('display.max_columns', 100)

# RMSE 계산 함수: sklearn 버전에 따라 mean_squared_error(squared=False)가 안 될 수 있어 직접 sqrt 처리합니다.
def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# 소수점 셋째 자리에서 버림 → 둘째 자리까지 출력할 때 사용
# 예: 1.678 -> 1.67
def floor_2(x):
    return math.floor(x * 100) / 100

# 데이터 폴더 자동 탐색
# - ChatGPT 환경: /mnt/data
# - 노트북과 CSV가 같은 폴더: 현재 폴더
# - 노트북 옆 dataset 폴더
# - 사용자가 말한 Windows 경로
CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / 'dataset',
    Path.cwd().parent / 'dataset',
    Path('/mnt/data'),
    Path(r'X:\study\docs\data_science\00_Practice\dataset'),
]

REQUIRED_FILES = [
    'TV.csv',
    'galaxy_users.csv',
    'mobiles.csv',
    'sales_pos.csv',
    'card_cust.csv',
    'edu_enrollees.csv',
]

DATA_DIR = None
for candidate in CANDIDATE_DIRS:
    if all((candidate / f).exists() for f in REQUIRED_FILES):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        'CSV 파일을 찾지 못했습니다. 노트북과 CSV 파일을 같은 폴더에 두거나 DATA_DIR 경로를 직접 지정하세요.'
    )

print('데이터 폴더:', DATA_DIR)

데이터 폴더: /mnt/data


---

# Set 01. 시중 TV 제품별 분석

사용 데이터: `TV.csv`

핵심 포인트는 **문자열 안에서 필요한 정보만 뽑아내는 것**입니다.

- 주사율: `60 Hz`, `120 Hz`처럼 숫자 + Hz 패턴 찾기
- 해상도: `HD`, `4K`, `8K` 찾기
- 변수 생성: 후기 작성 비율, 할인율, OTT 제공 여부 등 만들기

In [2]:
tv = pd.read_csv(DATA_DIR / 'TV.csv')
print(tv.shape)
tv.head()

(666, 11)


,Product_Name,Stars,Ratings,Reviews,current_price,MRP,channel,Operating_system,Picture_quality,Speaker,Frequency
0,Croma,4.2,1773,217,7990,20000,HD Ready 1366 x 768 Pixels,20 Speaker Output,60 Hz Refresh Rate,2 x HDMI | 2 x USB,1 Year Warranty
1,Adsun,3.8,6742,930,8699,21999,Netflix|Disney+Hotstar|Youtube,Operating System: Android Based,HD Ready 1366 x 768 Pixels,20 W Speaker Output,60 Hz Refresh Rate
2,LG,4.4,38870,3443,16499,21990,Netflix|Prime Video|Disney+Hotstar|Youtube,Operating System: WebOS,HD Ready 1366 x 768 Pixels,10 W Speaker Output,50 Hz Refresh Rate
3,OnePlus,4.3,101256,9189,16499,21999,Netflix|Prime Video|Disney+Hotstar|Youtube,Operating System: Android,HD Ready 1366 x 768 Pixels,20 W Speaker Output,60 Hz Refresh Rate
4,Xiaomi,4.3,3120,305,15499,24999,Netflix|Prime Video|Disney+Hotstar|Youtube,Operating System: Android,HD Ready 1366 x 768 Pixels,20 W Speaker Output,60 Hz Refresh Rate


## Set 01 - Q01

### 문제 핵심

주사율 정보가 `Frequency`, `Picture_quality`, `Speaker` 세 컬럼에 흩어져 있습니다.

✅ 해야 할 일:

1. 세 컬럼을 차례대로 확인합니다.
2. `2~3자리 숫자 + Hz` 패턴을 찾습니다.
3. 찾은 주사율이 `60`인 TV 개수를 셉니다.

### 정규표현식 설명

```python
r'(\d{2,3})\s*Hz'
```

- `\d{2,3}`: 숫자가 2자리 또는 3자리
- `\s*`: 숫자와 Hz 사이 공백이 있어도 되고 없어도 됨
- `Hz`: 단위

In [3]:
def extract_hz(row):
    # 문제에서 주사율이 흩어져 있다고 한 세 컬럼을 확인합니다.
    for col in ['Frequency', 'Picture_quality', 'Speaker']:
        text = str(row[col])
        matched = re.search(r'(\d{2,3})\s*Hz', text, flags=re.IGNORECASE)
        if matched:
            return int(matched.group(1))
    return np.nan

tv_q1 = tv.copy()
tv_q1['frequency_hz'] = tv_q1.apply(extract_hz, axis=1)

print(tv_q1['frequency_hz'].value_counts(dropna=False).sort_index())
answer_1_1 = int((tv_q1['frequency_hz'] == 60).sum())
print('✅ Set 01 Q01 정답:', answer_1_1)

frequency_hz
50.0      69
58.0       1
60.0     510
100.0     30
120.0     30
200.0     19
300.0      1
800.0      2
NaN        4
Name: count, dtype: int64
✅ Set 01 Q01 정답: 510


## Set 01 - Q02

### 문제 핵심

해상도 정보가 `Operating_system`, `channel`, `Picture_quality` 세 컬럼에 흩어져 있습니다.

주의할 점은 `4K Ultra HD` 안에도 `HD`라는 글자가 들어간다는 것입니다.  
그래서 우선순위를 **8K → 4K → HD** 순서로 둡니다.

✅ 계산식:

\[
|\text{8K 제품의 Stars 평균} - \text{4K 제품의 Stars 평균}|
\]

In [4]:
def extract_resolution(row):
    text = ' '.join(str(row[col]) for col in ['Operating_system', 'channel', 'Picture_quality'])
    text = text.upper()
    
    # 4K Ultra HD 같은 문구 때문에 HD보다 4K/8K를 먼저 확인합니다.
    if '8K' in text:
        return '8K'
    elif '4K' in text:
        return '4K'
    elif 'HD' in text:
        return 'HD'
    else:
        return np.nan

tv_q2 = tv.copy()
tv_q2['resolution'] = tv_q2.apply(extract_resolution, axis=1)

mean_by_resolution = tv_q2.groupby('resolution')['Stars'].mean()
display(mean_by_resolution)

answer_1_2 = round(abs(mean_by_resolution['8K'] - mean_by_resolution['4K']), 2)
print('✅ Set 01 Q02 정답:', answer_1_2)

resolution
4K    3.218082
8K    3.600000
HD    3.074333
Name: Stars, dtype: float64

✅ Set 01 Q02 정답: 0.38


## Set 01 - Q03

### 문제 핵심

Random Forest 회귀 모델을 만들고, 변수 중요도(`feature_importances_`)가 가장 큰 변수를 찾습니다.

✅ 독립변수 생성:

| 변수 | 계산 방법 |
|---|---|
| 후기작성비율 | `Reviews / Ratings` |
| 공장출고가 | `MRP` |
| 할인율 | `current_price / MRP` |
| Netflix제공여부 | `channel`에 Netflix가 있으면 1 |
| PrimeVideo제공여부 | `channel`에 Prime이 있으면 1 |
| 고해상도여부 | `Picture_quality`에 4K 또는 8K가 있으면 1 |

✅ 제외 조건:

`channel`에 `Pixel` 또는 `Oper` 문자가 있는 행은 제외합니다.

In [5]:
tv_q3 = tv[~tv['channel'].str.contains('Pixel|Oper', case=False, na=False)].copy()

# 파생변수 생성
tv_q3['후기작성비율'] = tv_q3['Reviews'] / tv_q3['Ratings']
tv_q3['공장출고가'] = tv_q3['MRP']
tv_q3['할인율'] = tv_q3['current_price'] / tv_q3['MRP']
tv_q3['Netflix제공여부'] = tv_q3['channel'].str.contains('Netflix', case=False, na=False).astype(int)
tv_q3['PrimeVideo제공여부'] = tv_q3['channel'].str.contains('Prime', case=False, na=False).astype(int)
tv_q3['고해상도여부'] = tv_q3['Picture_quality'].str.contains('4K|8K', case=False, na=False).astype(int)

feature_cols = ['후기작성비율', '공장출고가', '할인율', 'Netflix제공여부', 'PrimeVideo제공여부', '고해상도여부']
target_col = 'Stars'

model_data = tv_q3[feature_cols + [target_col]].replace([np.inf, -np.inf], np.nan).dropna()
print('학습 대상 행 개수:', len(model_data))

X = model_data[feature_cols]
y = model_data[target_col]

rf = RandomForestRegressor(random_state=123)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
display(importance)

answer_1_3 = importance.idxmax()
print('✅ Set 01 Q03 정답:', answer_1_3)

학습 대상 행 개수: 197


할인율               0.444691
후기작성비율            0.312206
공장출고가             0.173840
Netflix제공여부       0.027141
PrimeVideo제공여부    0.024004
고해상도여부            0.018118
dtype: float64

✅ Set 01 Q03 정답: 할인율


---

# Set 02. 갤럭시 사용자 데이터 분석

사용 데이터: `galaxy_users.csv`

핵심 포인트는 **Yes/No 범주형 값을 숫자로 바꾸는 것**입니다.

- Yes → 1
- No → 0
- 그 외 값은 문제 조건에 따라 제거하거나 -1 처리

In [6]:
galaxy = pd.read_csv(DATA_DIR / 'galaxy_users.csv')
print(galaxy.shape)
galaxy.head()

(7032, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Set 02 - Q01

### 문제 핵심

부가서비스 6개 컬럼에서 `Yes` 개수를 세면 고객별 부가서비스 사용 개수가 됩니다.

✅ 계산 절차:

1. 부가서비스 컬럼 6개 선택
2. 모든 값이 `Yes` 또는 `No`인 행만 남김
3. `Yes=1`, `No=0` 변환
4. 행별 합계로 부가서비스 개수 계산
5. `1개 사용 고객 수 / 6개 사용 고객 수` 계산

In [7]:
service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

galaxy_q1 = galaxy.copy()
valid_mask = galaxy_q1[service_cols].isin(['Yes', 'No']).all(axis=1)
galaxy_q1 = galaxy_q1[valid_mask].copy()

galaxy_q1[service_cols] = galaxy_q1[service_cols].replace({'Yes': 1, 'No': 0})
galaxy_q1['service_count'] = galaxy_q1[service_cols].sum(axis=1)

count_by_service = galaxy_q1['service_count'].value_counts().sort_index()
display(count_by_service)

answer_2_1 = round(count_by_service.loc[1] / count_by_service.loc[6], 1)
print('✅ Set 02 Q01 정답:', answer_2_1)

/tmp/ipykernel_15709/1241206786.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  galaxy_q1[service_cols] = galaxy_q1[service_cols].replace({'Yes': 1, 'No': 0})


service_count
0     693
1     966
2    1033
3    1117
4     850
5     569
6     284
Name: count, dtype: int64

✅ Set 02 Q01 정답: 3.4


## Set 02 - Q02

### 문제 핵심

상관분석은 두 숫자형 변수의 관계가 얼마나 강한지 보는 방법입니다.

통신사 사용 월수는 다음처럼 계산합니다.

\[
\text{통신사 사용 월수} = \left\lfloor \frac{\text{TotalCharges}}{\text{MonthlyCharges}} \right\rfloor
\]

여기서 `몫`이라고 했으므로 소수점 아래는 버립니다.

✅ 주의:

상관행렬에서 자기 자신과의 상관계수는 항상 1입니다.  
따라서 대각선 값은 제외하고 가장 큰 절대값을 찾습니다.

In [8]:
galaxy_q2 = galaxy[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
galaxy_q2['통신사사용월수'] = np.floor(galaxy_q2['TotalCharges'] / galaxy_q2['MonthlyCharges'])

corr_matrix = galaxy_q2[['tenure', 'MonthlyCharges', '통신사사용월수']].corr()
display(corr_matrix)

# 자기 자신과의 상관계수 1은 제외합니다.
abs_corr = corr_matrix.abs()
np.fill_diagonal(abs_corr.values, np.nan)

answer_2_2 = round(abs_corr.max().max(), 3)
print('✅ Set 02 Q02 정답:', answer_2_2)

,tenure,MonthlyCharges,통신사사용월수
tenure,1.000000,0.246862,0.998832
MonthlyCharges,0.246862,1.000000,0.246149
통신사사용월수,0.998832,0.246149,1.000000


✅ Set 02 Q02 정답: 0.999


## Set 02 - Q03

### 문제 핵심

고객 이탈 여부 `Churn`을 예측하는 로지스틱 회귀 모델을 만듭니다.

✅ 절차:

1. 문제에서 지정한 독립변수만 선택
2. 범주형 변수는 `Yes=1`, `No=0`, 그 외는 `-1`
3. Train/Test를 7:3으로 분리
4. Train 데이터로 MinMaxScaler를 학습
5. Test 데이터는 Train 기준으로 변환
6. Logistic Regression 학습
7. F1-score 계산

### F1-score 의미

F1-score는 정밀도와 재현율의 균형을 보는 지표입니다.  
이탈 고객 예측처럼 클래스 불균형이 있을 수 있는 문제에서 자주 사용합니다.

In [9]:
feature_cols = [
    'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'MonthlyCharges', 'TotalCharges',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingMovies', 'PaperlessBilling'
]

galaxy_q3 = galaxy[feature_cols + ['Churn']].copy()

binary_cols = [
    'Partner', 'Dependents', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingMovies',
    'PaperlessBilling', 'Churn'
]

for col in binary_cols:
    galaxy_q3[col] = galaxy_q3[col].map({'Yes': 1, 'No': 0}).fillna(-1)

galaxy_q3 = galaxy_q3.dropna()

X = galaxy_q3[feature_cols]
y = galaxy_q3['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logit = LogisticRegression(random_state=123, max_iter=1000)
logit.fit(X_train_scaled, y_train)

pred = logit.predict(X_test_scaled)
f1 = f1_score(y_test, pred)

answer_2_3 = round(f1, 2)
print('F1-score 원값:', f1)
print('✅ Set 02 Q03 정답:', answer_2_3)

F1-score 원값: 0.5479723046488625
✅ Set 02 Q03 정답: 0.55


---

# Set 03. 시중 스마트폰 상세 정보

사용 데이터: `mobiles.csv`

핵심 포인트는 이상치 판정, 상관분석, k-NN 회귀 모델 비교입니다.

In [10]:
mobile = pd.read_csv(DATA_DIR / 'mobiles.csv')
print(mobile.shape)
mobile.head()

(430, 11)


,screen_size,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,ratings,num_of_ratings,sales_price,discount_percent,sales
0,Very Small,64,2,1,1,1800,4.5,38645,32999,0.17,127.52
1,Small,64,4,2,1,2815,4.5,244,57149,0.04,1.39
2,Very Small,64,2,1,1,1800,4.5,38645,32999,0.17,127.52
3,Medium,64,3,1,1,2942,4.6,5366,42999,0.10,23.07
4,Medium,128,4,2,1,2815,4.6,745,69149,0.02,5.15


## Set 03 - Q01

### 문제 핵심

판매지수 `sales`가 평균보다 많이 높은 제품을 이상치, 즉 **주목받는 제품**으로 봅니다.

문제 조건:

\[
\text{이상치 기준} = \text{sales 평균} + 2 \times \text{sales 표준편차}
\]

성능지표는 다음 식으로 계산합니다.

\[
E = \frac{ROM}{32} + \frac{RAM}{2} + \text{카메라 개수} + \frac{battery\_capacity}{1000}
\]

여기서 카메라 개수는 다음과 같습니다.

\[
\text{카메라 개수} = \text{num\_rear\_camera} + \text{num\_front\_camera}
\]

In [11]:
mobile_q1 = mobile.copy()

sales_threshold = mobile_q1['sales'].mean() + 2 * mobile_q1['sales'].std()
attention_products = mobile_q1[mobile_q1['sales'] > sales_threshold].copy()

attention_products['E'] = (
    attention_products['ROM'] / 32
    + attention_products['RAM'] / 2
    + attention_products['num_rear_camera']
    + attention_products['num_front_camera']
    + attention_products['battery_capacity'] / 1000
)

print('이상치 기준:', sales_threshold)
print('주목받는 제품 수:', len(attention_products))
display(attention_products[['ROM', 'RAM', 'num_rear_camera', 'num_front_camera', 'battery_capacity', 'sales', 'E']].head())

answer_3_1 = round(attention_products['E'].mean(), 2)
print('✅ Set 03 Q01 정답:', answer_3_1)

이상치 기준: 146.55150129273215
주목받는 제품 수: 16


,ROM,RAM,num_rear_camera,num_front_camera,battery_capacity,sales,E
98,128,6,2,1,4000,231.79,14.0
110,128,6,4,2,4500,427.22,17.5
158,32,2,2,1,5000,167.73,10.0
159,32,2,2,1,5000,167.73,10.0
193,16,2,2,1,4000,147.52,8.5


✅ Set 03 Q01 정답: 11.01


## Set 03 - Q02

### 문제 핵심

후면 카메라가 1개인 제품은 제외하고, 지정된 변수들과 `sales` 간 피어슨 상관계수를 계산합니다.

✅ 찾을 값:

상관계수의 **절대값**이 가장 큰 변수의 상관계수

즉, `-0.9`와 `0.8`이 있으면 절대값 기준으로 `-0.9`가 더 큽니다.

In [12]:
mobile_q2 = mobile[mobile['num_rear_camera'] != 1].copy()

corr_cols = [
    'battery_capacity', 'ratings', 'num_of_ratings',
    'sales_price', 'discount_percent', 'sales'
]

corr_with_sales = mobile_q2[corr_cols].corr()['sales'].drop('sales')
display(corr_with_sales)

max_corr_var = corr_with_sales.abs().idxmax()
answer_3_2 = round(corr_with_sales[max_corr_var], 2)

print('절대값 기준 가장 큰 변수:', max_corr_var)
print('✅ Set 03 Q02 정답:', answer_3_2)

battery_capacity    0.025680
ratings             0.226075
num_of_ratings      0.949114
sales_price        -0.247760
discount_percent    0.223471
Name: sales, dtype: float64

절대값 기준 가장 큰 변수: num_of_ratings
✅ Set 03 Q02 정답: 0.95


## Set 03 - Q03

### 문제 핵심

`sales`를 예측하는 k-NN 회귀 모델을 만들고, k값별 RMSE를 비교합니다.

✅ 절차:

1. `sales`를 제외한 모든 변수를 독립변수로 사용
2. 명목형 변수인 `screen_size`는 One-Hot Encoding
3. Train/Test를 8:2로 분리
4. MinMax 정규화
5. k = 3, 5, 7, 9, 11 모델을 각각 학습
6. RMSE가 가장 낮은 k 선택

### RMSE 의미

\[
RMSE = \sqrt{\frac{1}{n}\sum (y_i - \hat{y_i})^2}
\]

예측값과 실제값 차이가 작을수록 RMSE가 낮습니다.  
따라서 **RMSE가 가장 작은 모델이 가장 좋은 모델**입니다.

In [13]:
X = pd.get_dummies(mobile.drop(columns='sales'), drop_first=False)
y = mobile['sales']

print('학습에 사용하는 독립변수 개수:', X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rmse_by_k = {}
for k in [3, 5, 7, 9, 11]:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    pred = knn.predict(X_test_scaled)
    rmse_by_k[k] = rmse_score(y_test, pred)

rmse_result = pd.Series(rmse_by_k, name='RMSE')
display(rmse_result)

answer_3_3 = int(rmse_result.idxmin())
print('✅ Set 03 Q03 정답:', answer_3_3)

학습에 사용하는 독립변수 개수: 14


3     40.440549
5     48.800827
7     53.186755
9     55.484384
11    56.160703
Name: RMSE, dtype: float64

✅ Set 03 Q03 정답: 3


---

# Set 04. 디지털프라자 매출 데이터

사용 데이터: `sales_pos.csv`

데이터가 55만 행이라 다른 데이터보다 큽니다.  
핵심 포인트는 고객 단위, 상품 단위로 `groupby` 집계를 하는 것입니다.

In [14]:
pos = pd.read_csv(DATA_DIR / 'sales_pos.csv')
print(pos.shape)
pos.head()

(550068, 11)


,user,prod,gender,age_group,job,city,marital,prod_cat1,prod_cat2,prod_cat3,purchase
0,1,P00069042,F,0-17,10,A,0,3,NaN,NaN,8370
1,1,P00248942,F,0-17,10,A,0,1,6.0,14.0,15200
2,1,P00087842,F,0-17,10,A,0,12,NaN,NaN,1422
3,1,P00085442,F,0-17,10,A,0,12,14.0,NaN,1057
4,2,P00285442,M,55+,16,C,0,8,NaN,NaN,7969


## Set 04 - Q01

### 문제 핵심

1행이 물품 1개 구매 내역입니다.

✅ 절차:

1. 상품(`prod`)별 결제금액(`purchase`) 합계 계산
2. 합계가 가장 큰 상품 찾기
3. 그 상품을 구매한 행만 필터링
4. 그중 가장 많이 등장한 직업(`job`) 번호 찾기

In [15]:
prod_purchase_sum = pos.groupby('prod')['purchase'].sum()
top_prod = prod_purchase_sum.idxmax()

job_count_for_top_prod = pos[pos['prod'] == top_prod]['job'].value_counts()

display(job_count_for_top_prod.head())
answer_4_1 = int(job_count_for_top_prod.idxmax())

print('매출액이 가장 큰 상품:', top_prod)
print('✅ Set 04 Q01 정답:', answer_4_1)

job
4     221
7     187
0     179
17    143
1     124
Name: count, dtype: int64

매출액이 가장 큰 상품: P00025442
✅ Set 04 Q01 정답: 4


## Set 04 - Q02

### 문제 핵심

26-35세 고객을 대상으로 결혼 여부별 구매 카테고리 다양성 차이를 봅니다.

카테고리 조합 예시:

```text
prod_cat1=1, prod_cat2=2, prod_cat3=0 → 1-2-0
```

✅ 절차:

1. `age_group == '26-35'` 필터링
2. `prod_cat1`, `prod_cat2`, `prod_cat3` 결측값을 0으로 대체
3. 세 카테고리를 문자열로 합쳐 하나의 카테고리 조합 생성
4. 고객별 서로 다른 카테고리 조합 개수 계산
5. 결혼 여부별 평균 계산
6. 두 평균 차이의 절대값 계산

In [16]:
pos_q2 = pos[pos['age_group'] == '26-35'].copy()

for col in ['prod_cat1', 'prod_cat2', 'prod_cat3']:
    pos_q2[col] = pos_q2[col].fillna(0).astype(int).astype(str)

pos_q2['category_combo'] = (
    pos_q2['prod_cat1'] + '-' + pos_q2['prod_cat2'] + '-' + pos_q2['prod_cat3']
)

user_marital = pos_q2[['user', 'marital']].drop_duplicates('user')
user_category_count = (
    pos_q2.groupby('user')['category_combo']
    .nunique()
    .reset_index(name='category_count')
    .merge(user_marital, on='user', how='left')
)

mean_by_marital = user_category_count.groupby('marital')['category_count'].mean()
display(mean_by_marital)

answer_4_2 = round(abs(mean_by_marital.loc[1] - mean_by_marital.loc[0]), 2)
print('✅ Set 04 Q02 정답:', answer_4_2)

marital
0    41.663183
1    41.792336
Name: category_count, dtype: float64

✅ Set 04 Q02 정답: 0.13


## Set 04 - Q03

### 문제 핵심

고객 1명당 1행이 되도록 고객 단위 데이터를 만든 뒤 K-means 군집분석을 합니다.

✅ 사용할 변수:

| 변수 | 만드는 방법 |
|---|---|
| 성별 | M=1, F=0 |
| 구매 상품 종류수 | 고객별 서로 다른 `prod` 개수 |
| 나이 | 순서형 0~6 |
| 직업 | One-Hot Encoding |
| 총 구매금액 | 고객별 `purchase` 합계 |
| 도시 | One-Hot Encoding |
| 결혼 여부 | 원래 값 사용 |

✅ 모델 조건:

- KMeans K=7
- MinMax 정규화
- seed=123
- Silhouette score 계산

### Silhouette score 의미

군집이 얼마나 잘 나뉘었는지 보는 지표입니다.  
값이 높을수록 군집 간 분리가 잘 된 편입니다.

In [17]:
user_profile = pos.groupby('user').agg(
    gender=('gender', 'first'),
    product_count=('prod', 'nunique'),
    age_group=('age_group', 'first'),
    job=('job', 'first'),
    total_purchase=('purchase', 'sum'),
    city=('city', 'first'),
    marital=('marital', 'first')
).reset_index()

age_order = ['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+']
age_map = {age: idx for idx, age in enumerate(age_order)}

user_profile['gender'] = user_profile['gender'].map({'M': 1, 'F': 0})
user_profile['age'] = user_profile['age_group'].map(age_map)

X = user_profile[[
    'gender', 'product_count', 'age', 'job',
    'total_purchase', 'city', 'marital'
]].copy()

X = pd.get_dummies(X, columns=['job', 'city'], drop_first=False)

print('고객 수:', len(user_profile))
print('사용 변수 수:', X.shape[1])

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=7, random_state=123, n_init=10)
cluster = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, cluster)
answer_4_3 = round(sil, 2)

print('Silhouette score 원값:', sil)
print('✅ Set 04 Q03 정답:', answer_4_3)

고객 수: 5891
사용 변수 수: 29


Silhouette score 원값: 0.17776082486017428
✅ Set 04 Q03 정답: 0.18


---

# Set 05. 신용카드 고객정보 분석

사용 데이터: `card_cust.csv`

문제 풀이 전에 `MINIMUM_PAYMENTS` 결측값을 평균으로 대체한 `base` 데이터를 만듭니다.

In [18]:
card = pd.read_csv(DATA_DIR / 'card_cust.csv')
print(card.shape)
card.head()

(1000, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0.0,2.0,1000.0,201.802084,139.509787,0.000000,12.0
1,10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4.0,0.0,7000.0,4103.032597,1072.340217,0.222222,12.0
2,10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0.0,12.0,7500.0,622.066742,627.284787,0.000000,12.0
3,10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1.0,1.0,7500.0,0.000000,NaN,0.000000,12.0
4,10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0.0,1.0,1200.0,678.334763,244.791237,0.000000,12.0


In [19]:
base_card = card.copy()
base_card['MINIMUM_PAYMENTS'] = base_card['MINIMUM_PAYMENTS'].fillna(
    base_card['MINIMUM_PAYMENTS'].mean()
)

print('MINIMUM_PAYMENTS 결측치 수:', base_card['MINIMUM_PAYMENTS'].isna().sum())

MINIMUM_PAYMENTS 결측치 수: 0


## Set 05 - Q01

### 문제 핵심

`TENURE`별로 `BALANCE`와 `CREDIT_LIMIT` 간 피어슨 상관계수를 구하고, 그중 가장 큰 값을 찾습니다.

✅ 절차:

1. `TENURE`별 그룹 생성
2. 각 그룹에서 `BALANCE`와 `CREDIT_LIMIT` 상관계수 계산
3. 가장 큰 상관계수 선택

In [20]:
corr_by_tenure = base_card.groupby('TENURE')[['BALANCE', 'CREDIT_LIMIT']].apply(
    lambda g: g['BALANCE'].corr(g['CREDIT_LIMIT'])
)

display(corr_by_tenure)

answer_5_1 = round(corr_by_tenure.max(), 2)
print('✅ Set 05 Q01 정답:', answer_5_1)

TENURE
6.0     0.868056
7.0     0.948405
8.0     0.820696
9.0     0.085474
10.0    0.291482
11.0    0.380360
12.0    0.460833
dtype: float64

✅ Set 05 Q01 정답: 0.95


## Set 05 - Q02

### 문제 핵심

고객 ID를 제외한 17개 변수를 Z-score 표준화한 뒤, K=2~5 중 Silhouette score가 가장 높은 K를 고릅니다.

그 다음 선택된 군집 기준으로 원본 데이터의 `ONEOFF_PURCHASES` 평균을 계산합니다.

✅ 주의:

군집분석 입력은 표준화된 데이터지만, 마지막 평균 계산은 **정규화하지 않은 원본 기준**입니다.

In [21]:
feature_cols = [col for col in base_card.columns if col != 'CUST_ID']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(base_card[feature_cols])

silhouette_by_k = {}
labels_by_k = {}

for k in range(2, 6):
    kmeans = KMeans(n_clusters=k, random_state=1234, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    silhouette_by_k[k] = silhouette_score(X_scaled, labels)
    labels_by_k[k] = labels

silhouette_result = pd.Series(silhouette_by_k, name='silhouette_score')
display(silhouette_result)

best_k = int(silhouette_result.idxmax())
base_card_q2 = base_card.copy()
base_card_q2['cluster'] = labels_by_k[best_k]

oneoff_mean_by_cluster = base_card_q2.groupby('cluster')['ONEOFF_PURCHASES'].mean()
display(oneoff_mean_by_cluster)

answer_5_2 = round(oneoff_mean_by_cluster.max(), 2)

print('최적 K:', best_k)
print('✅ Set 05 Q02 정답:', answer_5_2)

2    0.307528
3    0.196361
4    0.207287
5    0.193500
Name: silhouette_score, dtype: float64

cluster
0     340.230998
1    3946.187525
Name: ONEOFF_PURCHASES, dtype: float64

최적 K: 2
✅ Set 05 Q02 정답: 3946.19


## Set 05 - Q03

### 문제 핵심

고객 ID가 4의 배수인지 여부로 Train/Test를 나눕니다.

- 4의 배수가 아닌 고객: Train
- 4의 배수인 고객: Test

종속변수는 `ONEOFF_PURCHASES`이고, 모델은 Decision Tree Regressor입니다.

✅ RMSE:

\[
B = \left( \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y_i})^2 \right)^{1/2}
\]

정답표에는 `1039.2`로 표기되어 있지만, 소수점 둘째 자리까지 계산하면 `1039.19`입니다.

In [22]:
train = base_card[base_card['CUST_ID'] % 4 != 0].copy()
test = base_card[base_card['CUST_ID'] % 4 == 0].copy()

feature_cols = [col for col in base_card.columns if col not in ['CUST_ID', 'ONEOFF_PURCHASES']]

X_train = train[feature_cols]
y_train = train['ONEOFF_PURCHASES']
X_test = test[feature_cols]
y_test = test['ONEOFF_PURCHASES']

tree = DecisionTreeRegressor(random_state=1234)
tree.fit(X_train, y_train)

pred = tree.predict(X_test)
rmse = rmse_score(y_test, pred)

answer_5_3_two_digits = round(rmse, 2)
answer_5_3_one_digit = round(rmse, 1)

print('RMSE 원값:', rmse)
print('소수점 둘째 자리:', answer_5_3_two_digits)
print('정답표 표기 형태:', answer_5_3_one_digit)
print('✅ Set 05 Q03 정답:', answer_5_3_two_digits)

RMSE 원값: 1039.193967231063
소수점 둘째 자리: 1039.19
정답표 표기 형태: 1039.2
✅ Set 05 Q03 정답: 1039.19


---

# Set 06. 교육 수강자 분석

사용 데이터: `edu_enrollees.csv`

먼저 문제에서 지정한 전처리 4단계를 수행해 `base` 데이터를 만듭니다.

## 전처리 4단계

1. `city`, `company_size`, `company_type` 제거
2. 결측치가 있는 행 제거
3. `experience`에서 `>20`, `<1` 제거 후 정수 변환
4. `last_new_job`에서 `>4`, `never` 제거 후 정수 변환

In [23]:
edu = pd.read_csv(DATA_DIR / 'edu_enrollees.csv')
print(edu.shape)
edu.head()

(19158, 15)


,enrollee_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target,Xgrp
0,8949.0,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36.0,1.0,train
1,29725.0,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47.0,0.0,train
2,11561.0,city_21,0.624,NaN,No relevant experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83.0,0.0,train
3,33241.0,city_115,0.789,NaN,No relevant experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52.0,1.0,train
4,666.0,city_162,0.767,Male,Has relevant experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8.0,0.0,train


In [24]:
base_edu = edu.drop(columns=['city', 'company_size', 'company_type']).dropna().copy()

base_edu = base_edu[~base_edu['experience'].isin(['>20', '<1'])].copy()
base_edu['experience'] = base_edu['experience'].astype(int)

base_edu = base_edu[~base_edu['last_new_job'].isin(['>4', 'never'])].copy()
base_edu['last_new_job'] = base_edu['last_new_job'].astype(int)

print('전처리 후 행/열:', base_edu.shape)
base_edu.head()

전처리 후 행/열: (7522, 12)


,enrollee_id,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,last_new_job,training_hours,target,Xgrp
8,27107.0,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,7,1,46.0,1.0,train
11,23853.0,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,5,1,108.0,0.0,train
19,11399.0,0.827,Female,Has relevant experience,no_enrollment,Graduate,Arts,4,1,132.0,1.0,train
20,31972.0,0.843,Male,Has relevant experience,no_enrollment,Masters,STEM,11,1,68.0,0.0,train
21,19061.0,0.926,Male,Has relevant experience,no_enrollment,Masters,STEM,11,2,50.0,0.0,train


## Set 06 - Q01

### 문제 핵심

관련 경험이 없는 사람과 있는 사람 각각에서 `target=1`, 즉 전배 희망 비율을 계산합니다.

\[
A = P(target=1 \mid \text{No relevant experience})
\]

\[
B = P(target=1 \mid \text{Has relevant experience})
\]

정답은 다음 비율입니다.

\[
\frac{A}{B}
\]

In [25]:
target_rate = base_edu.groupby('relevant_experience')['target'].mean()
display(target_rate)

A = target_rate['No relevant experience']
B = target_rate['Has relevant experience']

answer_6_1 = round(A / B, 2)
print('✅ Set 06 Q01 정답:', answer_6_1)

relevant_experience
Has relevant experience    0.215911
No relevant experience     0.382873
Name: target, dtype: float64

✅ Set 06 Q01 정답: 1.77


## Set 06 - Q02

### 문제 핵심

로지스틱 회귀분석을 하고, 각 변수의 Odds Ratio를 계산합니다.

### Odds Ratio란?

로지스틱 회귀 계수는 그대로 해석하기 어렵기 때문에 보통 지수 변환을 합니다.

\[
\text{Odds Ratio} = e^{\beta}
\]

- Odds Ratio > 1: 해당 변수가 커질수록 target=1 가능성이 증가
- Odds Ratio < 1: 해당 변수가 커질수록 target=1 가능성이 감소

✅ 주의:

- `enrollee_id`는 식별자라서 분석 변수에서 제외합니다.
- `Xgrp`는 Train/Test 구분용이라 회귀분석 변수에서는 제외합니다.
- 범주형 변수는 더미 변수화하고, 문제 조건대로 **마지막 범주를 제외**합니다.
- 결과는 소수점 셋째 자리에서 버림하여 둘째 자리까지 출력합니다.

In [26]:
def make_job2_dataset(base):
    """Set 06 Q02~Q03에서 사용할 job2 데이터셋 생성 함수"""
    # 분석 변수에서 식별자와 분리용 컬럼은 제외합니다.
    feature_cols = [
        col for col in base.columns
        if col not in ['target', 'Xgrp', 'enrollee_id']
    ]
    
    X_raw = base[feature_cols].copy().reset_index(drop=True)
    parts = []
    
    for col in X_raw.columns:
        if X_raw[col].dtype == 'object':
            dummy = pd.get_dummies(X_raw[col], prefix=col).astype(int)
            # 마지막 범주 제외
            dummy = dummy.iloc[:, :-1]
            parts.append(dummy)
        else:
            parts.append(X_raw[[col]])
    
    X = pd.concat(parts, axis=1)
    job2 = pd.concat(
        [
            X,
            base[['target', 'Xgrp']].reset_index(drop=True)
        ],
        axis=1
    )
    return job2

job2 = make_job2_dataset(base_edu)

X = job2.drop(columns=['target', 'Xgrp'])
y = job2['target']

logit = LogisticRegression(
    C=100000,
    max_iter=1000,
    solver='liblinear',
    random_state=123
)
logit.fit(X, y)

odds_ratio = pd.Series(np.exp(logit.coef_[0]), index=X.columns).sort_values(ascending=False)
display(odds_ratio.head(10))

max_odds = odds_ratio.max()
answer_6_2 = floor_2(max_odds)

print('가장 큰 Odds Ratio 원값:', max_odds)
print('가장 큰 변수:', odds_ratio.idxmax())
print('✅ Set 06 Q02 정답:', answer_6_2)

enrolled_university_Full time course    1.674323
major_discipline_No Major               1.499740
education_level_Graduate                1.383458
major_discipline_Arts                   1.312425
major_discipline_Humanities             1.280568
major_discipline_Business Degree        1.116581
last_new_job                            1.099749
education_level_Masters                 1.025129
training_hours                          0.999073
experience                              0.971781
dtype: float64

가장 큰 Odds Ratio 원값: 1.6743230951297654
가장 큰 변수: enrolled_university_Full time course
✅ Set 06 Q02 정답: 1.67


## Set 06 - Q03

### 문제 핵심

Q02에서 만든 `job2`를 사용합니다.

✅ 절차:

1. `Xgrp == 'train'`인 행을 Train으로 사용
2. `Xgrp == 'test'`인 행을 Test로 사용
3. `target`을 종속변수로 사용
4. KNeighborsClassifier, n_neighbors=5 학습
5. Accuracy 계산

문제에서 정규화 지시가 없으므로 여기서는 `job2` 값을 그대로 사용합니다.

\[
Accuracy = \frac{TP + TN}{Total}
\]

In [27]:
X = job2.drop(columns=['target', 'Xgrp'])
y = job2['target']

train_mask = job2['Xgrp'] == 'train'
test_mask = job2['Xgrp'] == 'test'

X_train = X.loc[train_mask]
y_train = y.loc[train_mask]
X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

pred = knn.predict(X_test)
acc = accuracy_score(y_test, pred)

answer_6_3 = round(acc, 2)
print('Accuracy 원값:', acc)
print('✅ Set 06 Q03 정답:', answer_6_3)

Accuracy 원값: 0.7134232954545454
✅ Set 06 Q03 정답: 0.71


---

# 최종 정리

## 📌 Set 01

- Q01: 주사율을 세 컬럼에서 정규표현식으로 추출 → `60Hz` 개수 **510**
- Q02: 해상도는 8K → 4K → HD 우선순위로 추출 → 평균 차이 **0.38**
- Q03: Random Forest 변수 중요도 최대 → **할인율**

## 📌 Set 02

- Q01: 부가서비스 1개 사용 고객 / 6개 사용 고객 → **3.4**
- Q02: 상관계수 절대값 최대 → **0.999**
- Q03: 로지스틱 회귀 F1-score → **0.55**

## 📌 Set 03

- Q01: 판매지수 이상치 제품의 성능지표 평균 → **11.01**
- Q02: sales와 절대상관이 가장 큰 변수의 상관계수 → **0.95**
- Q03: RMSE가 가장 낮은 k → **3**

## 📌 Set 04

- Q01: 최고 매출 상품을 가장 많이 구매한 직업 → **4**
- Q02: 결혼여부별 카테고리 개수 평균 차이 → **0.13**
- Q03: K=7 K-means Silhouette score → **0.18**

## 📌 Set 05

- Q01: TENURE별 BALANCE-CREDIT_LIMIT 상관계수 최대 → **0.95**
- Q02: 최적 K 군집의 ONEOFF_PURCHASES 평균 최대 → **3946.19**
- Q03: Decision Tree RMSE → **1039.19**  
  - 정답표 표기처럼 소수점 한 자리면 **1039.2**

## 📌 Set 06

- Q01: 관련 경험 없음/있음 전배 희망 비율의 비 → **1.77**
- Q02: Odds Ratio 최대값 → **1.67**
- Q03: KNN Accuracy → **0.71**